In [8]:
import pandas as pd
import numpy as np

# First check what sheets exist
xl = pd.ExcelFile("../data/raw/landslidedata.xlsx")
print("Sheets found:", xl.sheet_names)

Sheets found: ['KANGRA', 'MANDI', 'SHIMLA', 'UNA', 'SOLAN', 'HAMIRPUR', 'BILASPUR', 'SIRMAUR', 'KULLU', 'CHAMBA', 'KINAAUR', 'LAHUL']


In [6]:
import pandas as pd
import numpy as np

# Load and combine all district sheets
all_sheets = pd.read_excel("../data/raw/landslidedata.xlsx", sheet_name=None)

# Combine all sheets into one dataframe
df = pd.concat(all_sheets.values(), ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nRows per district:")
print(df['District'].value_counts())

Total rows: 3972580
Columns: ['Cordinates', 'Unnamed: 1', 'TEHSIL', 'District', 'STATE', 'Exposure', 'merged_column']

Rows per district:
District
KANGRA           980590
MANDI            468894
SHIMLA           441471
UNA              327574
SOLAN            316896
HAMIRPUR         305911
BILASPUR         251159
SIRMAUR          248289
KULLU            247465
CHAMBA           247269
KINNAUR          105262
LAHUL & SPITI     31800
Name: count, dtype: int64


In [9]:
df['clean_coords'] = df['Cordinates'].fillna(df['merged_column'])
df[['lat', 'lon']] = df['clean_coords'].str.split(',', expand=True).astype(float)
print(f"Null lats: {df['lat'].isna().sum()}")
print(f"Null lons: {df['lon'].isna().sum()}")

Null lats: 0
Null lons: 0


In [10]:
label_map = {
    'Not-exposed'        : 0,
    'Concerned'          : 1,
    'At Risk'            : 2,
    'Vulnerable'         : 3,
    'Severely Vulnerable': 4,
    'Highly Vulnerable'  : 5
}
df['label'] = df['Exposure'].map(label_map)
print(df['label'].value_counts().sort_index())

label
0.0    3393072
1.0     151979
2.0      77753
3.0     154932
4.0      81580
5.0     112477
Name: count, dtype: int64


In [11]:
df_sampled = df.groupby('Exposure', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42),
    include_groups=False
)
df_sampled['Exposure'] = df.loc[df_sampled.index, 'Exposure']
df_sampled['label'] = df_sampled['Exposure'].map(label_map)
df_sampled[['lat', 'lon']] = df.loc[df_sampled.index, ['lat', 'lon']].values

print(f"Sampled rows: {len(df_sampled)}")
print(df_sampled['label'].value_counts().sort_index())

Sampled rows: 12000
label
0    2000
1    2000
2    2000
3    2000
4    2000
5    2000
Name: count, dtype: int64


In [13]:
import os

# Create folders if they don't exist
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/raw", exist_ok=True)

print("Folders created!")

Folders created!


In [14]:
df_sampled.to_csv("../data/processed/sampled_hp.csv", index=False)
print("Saved successfully!")
print(df_sampled[['lat', 'lon', 'District', 'Exposure', 'label']].head(10))

Saved successfully!
               lat      lon District Exposure  label
27871    31.808141  76.1385   KANGRA  At Risk      2
3402675  31.660223  77.3072    KULLU  At Risk      2
1856814  31.459956  77.6727   SHIMLA  At Risk      2
1488138  31.077288  77.1815   SHIMLA  At Risk      2
3096238  30.558009  77.2922  SIRMAUR  At Risk      2
3562946  32.251112  77.1902    KULLU  At Risk      2
1499247  31.077502  77.1811   SHIMLA  At Risk      2
2214983  31.791580  76.1241      UNA  At Risk      2
3255166  30.603124  77.6744  SIRMAUR  At Risk      2
2096261  31.590760  76.3474      UNA  At Risk      2


In [4]:
import os
for file in os.listdir("../data/raw/"):
    print(file)

northern-zone-260311-free.shp
output_SRTMGL1.tif
landslidedata.xlsx


In [5]:
for file in os.listdir("../data/raw/northern-zone-260311-free.shp/"):
    print(file)

gis_osm_natural_a_free_1.prj
gis_osm_places_a_free_1.dbf
gis_osm_places_free_1.prj
gis_osm_transport_free_1.dbf
gis_osm_pofw_a_free_1.prj
gis_osm_pois_free_1.prj
gis_osm_water_a_free_1.dbf
gis_osm_traffic_a_free_1.prj
gis_osm_pofw_free_1.dbf
gis_osm_traffic_free_1.prj
gis_osm_natural_free_1.cpg
gis_osm_landuse_a_free_1.shp
gis_osm_railways_free_1.prj
gis_osm_landuse_a_free_1.cpg
gis_osm_natural_free_1.shp
gis_osm_transport_a_free_1.prj
gis_osm_waterways_free_1.dbf
gis_osm_buildings_a_free_1.dbf
gis_osm_landuse_a_free_1.shx
gis_osm_natural_free_1.shx
gis_osm_buildings_a_free_1.shp
gis_osm_waterways_free_1.cpg
gis_osm_waterways_free_1.shp
gis_osm_buildings_a_free_1.cpg
gis_osm_pois_a_free_1.prj
gis_osm_pofw_free_1.shx
gis_osm_buildings_a_free_1.shx
gis_osm_waterways_free_1.shx
gis_osm_natural_free_1.dbf
gis_osm_landuse_a_free_1.dbf
gis_osm_pofw_free_1.cpg
gis_osm_roads_free_1.prj
gis_osm_pofw_free_1.shp
gis_osm_water_a_free_1.shp
gis_osm_water_a_free_1.cpg
gis_osm_transport_free_1.cpg
gi

In [12]:
import rasterio
from rasterio.merge import merge
import os

# Load both DEM files
dem1 = rasterio.open("../data/raw/output_SRTMGL1.tif")        # main HP
dem2 = rasterio.open("../data/raw/output_SRTMGL1_west.tif")   # western strip

# Merge into one
merged, merged_transform = merge([dem1, dem2])

# Save merged DEM
merged_meta = dem1.meta.copy()
merged_meta.update({
    "driver": "GTiff",
    "height": merged.shape[1],
    "width": merged.shape[2],
    "transform": merged_transform
})

with rasterio.open("../data/raw/hp_dem_full.tif", "w", **merged_meta) as dest:
    dest.write(merged)

dem1.close()
dem2.close()

print("✅ DEMs merged successfully!")

# Verify coverage
with rasterio.open("../data/raw/hp_dem_full.tif") as src:
    print(f"New bounds: {src.bounds}")
    print(f"Should now start from ~75.5 west")

✅ DEMs merged successfully!
New bounds: BoundingBox(left=75.42875000003383, bottom=30.2845833333294, right=79.17736111114544, top=33.64458333332985)
Should now start from ~75.5 west


In [17]:
import os
for file in os.listdir("../data/raw/northern.shp/"):
    if file.endswith('.shp'):
        print(file)

gis_osm_landuse_a_free_1.shp
gis_osm_natural_free_1.shp
gis_osm_buildings_a_free_1.shp
gis_osm_waterways_free_1.shp
gis_osm_pofw_free_1.shp
gis_osm_water_a_free_1.shp
gis_osm_transport_free_1.shp
gis_osm_places_a_free_1.shp
gis_osm_natural_a_free_1.shp
gis_osm_places_free_1.shp
gis_osm_pofw_a_free_1.shp
gis_osm_pois_free_1.shp
gis_osm_traffic_a_free_1.shp
gis_osm_traffic_free_1.shp
gis_osm_railways_free_1.shp
gis_osm_transport_a_free_1.shp
gis_osm_pois_a_free_1.shp
gis_osm_roads_free_1.shp


In [ ]:
for file in os.listdir("../data/raw/"):
    if file.endswith('.tif')
        print(file)

hp_dem_full.tif
output_SRTMGL1.tif
output_SRTMGL1_west.tif


In [19]:
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
import richdem as rd
import os
import time
import requests

/Users/tanmay/landslidehp/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [20]:
df_clean = pd.read_csv("../data/processed/sampled_hp.csv")
df_clean = df_clean[
    (df_clean['lon'] >= 75.428) & 
    (df_clean['lon'] <= 79.177) &
    (df_clean['lat'] >= 30.284) & 
    (df_clean['lat'] <= 33.644)
].copy().reset_index(drop=True)
print(f"✅ Working with {len(df_clean)} points")

✅ Working with 11225 points


In [21]:
dem_path = "../data/raw/hp_dem_full.tif"
coords   = list(zip(df_clean['lon'], df_clean['lat']))

with rasterio.open(dem_path) as src:
    df_clean['elevation'] = [val[0] for val in src.sample(coords)]

print(f"✅ Elevation extracted")
print(df_clean['elevation'].describe())

✅ Elevation extracted
count    11225.000000
mean      1372.358129
std        559.113078
min        264.000000
25%        945.000000
50%       1309.000000
75%       1768.000000
max       4624.000000
Name: elevation, dtype: float64


In [23]:
import numpy as np
import rasterio
from scipy.ndimage import generic_filter

with rasterio.open(dem_path) as src:
    dem = src.read(1).astype(float)
    meta = src.meta.copy()
    res = src.res
    nodata = src.nodata
    if nodata:
        dem[dem == nodata] = np.nan

x_res = res[0] * 111320
y_res = res[1] * 111320

dy, dx = np.gradient(dem, y_res, x_res)

slope     = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
aspect    = np.degrees(np.arctan2(-dx, dy)) % 360
roughness = generic_filter(
    dem,
    lambda x: np.nanmax(x) - np.nanmin(x),
    size=3
)
print("✅ Slope, Aspect, Roughness computed")

meta.update(dtype='float32', count=1)

with rasterio.open("../data/raw/hp_slope.tif", 'w', **meta) as dst:
    dst.write(slope.astype('float32'), 1)
print("✅ Slope saved")

with rasterio.open("../data/raw/hp_aspect.tif", 'w', **meta) as dst:
    dst.write(aspect.astype('float32'), 1)
print("✅ Aspect saved")

with rasterio.open("../data/raw/hp_roughness.tif", 'w', **meta) as dst:
    dst.write(roughness.astype('float32'), 1)
print("✅ Roughness saved")

/var/folders/wr/tfs09wcn2rqdx4n2g4y49jlw0000gn/T/ipykernel_42825/2698427533.py:22: RuntimeWarning: All-NaN slice encountered
  lambda x: np.nanmax(x) - np.nanmin(x),


✅ Slope, Aspect, Roughness computed
✅ Slope saved
✅ Aspect saved
✅ Roughness saved


In [24]:
for feature, path in [
    ('slope',     "../data/raw/hp_slope.tif"),
    ('aspect',    "../data/raw/hp_aspect.tif"),
    ('roughness', "../data/raw/hp_roughness.tif")
]:
    with rasterio.open(path) as src:
        df_clean[feature] = [val[0] for val in src.sample(coords)]
    print(f"✅ {feature} extracted")

print("\nDEM Summary:")
print(df_clean[['elevation','slope','aspect','roughness']].describe())


✅ slope extracted
✅ aspect extracted
✅ roughness extracted

DEM Summary:
          elevation         slope        aspect     roughness
count  11225.000000  11225.000000  11225.000000  11225.000000
mean    1372.358129     18.923109    180.902222     28.366146
std      559.113078      8.752042     91.012413     13.556648
min      264.000000      0.000000      0.000000      0.000000
25%      945.000000     13.035095    112.479431     19.000000
50%     1309.000000     19.525883    185.079605     28.000000
75%     1768.000000     25.004469    247.750977     37.000000
max     4624.000000     53.317665    358.698059     95.000000


In [25]:
osm_path  = "../data/raw/northern.shp/"

roads     = gpd.read_file(osm_path + "gis_osm_roads_free_1.shp")
buildings = gpd.read_file(osm_path + "gis_osm_buildings_a_free_1.shp")
landuse   = gpd.read_file(osm_path + "gis_osm_landuse_a_free_1.shp")
waterways = gpd.read_file(osm_path + "gis_osm_waterways_free_1.shp")

print(f"✅ OSM files loaded")
print(f"Roads:     {len(roads)} features")
print(f"Buildings: {len(buildings)} features")
print(f"Landuse:   {len(landuse)} features")
print(f"Waterways: {len(waterways)} features")

✅ OSM files loaded
Roads:     1857949 features
Buildings: 855374 features
Landuse:   120036 features
Waterways: 25637 features


In [26]:
points_gdf = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean.lon, df_clean.lat),
    crs="EPSG:4326"
).to_crs("EPSG:32644")

roads     = roads.to_crs("EPSG:32644")
buildings = buildings.to_crs("EPSG:32644")
landuse   = landuse.to_crs("EPSG:32644")
waterways = waterways.to_crs("EPSG:32644")

print("✅ All layers projected to meters")

✅ All layers projected to meters


In [29]:
from shapely.strtree import STRtree
import numpy as np

def strtree_distance(points_gdf, features_gdf):
    tree = STRtree(features_gdf.geometry.values)
    distances = []
    for point in points_gdf.geometry:
        nearest = tree.nearest(point)
        dist = point.distance(features_gdf.geometry.iloc[nearest])
        distances.append(dist)
    return distances

# Roads
print("Computing dist_to_road with STRtree...")
df_clean['dist_to_road'] = strtree_distance(points_gdf, roads)
print("✅ dist_to_road done")
print(df_clean['dist_to_road'].describe())

Computing dist_to_road with STRtree...
✅ dist_to_road done
count    11225.000000
mean       154.823896
std        257.486496
min          0.042840
25%         22.250688
50%         59.510658
75%        173.854188
max       5079.992528
Name: dist_to_road, dtype: float64


In [30]:
from shapely.strtree import STRtree
import numpy as np

def strtree_distance(points_gdf, features_gdf):
    tree = STRtree(features_gdf.geometry.values)
    distances = []
    for point in points_gdf.geometry:
        nearest = tree.nearest(point)
        dist = point.distance(features_gdf.geometry.iloc[nearest])
        distances.append(dist)
    return distances

# Distance to building
print("Computing dist_to_building...")
df_clean['dist_to_building'] = strtree_distance(points_gdf, buildings)
print("✅ dist_to_building done")

# Distance to river
print("Computing dist_to_river...")
df_clean['dist_to_river'] = strtree_distance(points_gdf, waterways)
print("✅ dist_to_river done")

# Building density (fast version using STRtree)
print("Computing building_density...")
tree_b = STRtree(buildings.geometry.values)
df_clean['building_density'] = [
    len(tree_b.query(p.buffer(1000)))
    for p in points_gdf.geometry
]
print("✅ building_density done")

# Landuse type (fast version)
print("Computing landuse_type...")
tree_l = STRtree(landuse.geometry.values)
df_clean['landuse_type'] = [
    landuse.iloc[tree_l.nearest(p)]['fclass']
    for p in points_gdf.geometry
]
print("✅ landuse_type done")

# Save progress immediately
df_clean.to_csv("../data/processed/features_progress.csv", index=False)

print("\n✅ ALL OSM FEATURES DONE!")
print(df_clean[['dist_to_road','dist_to_building',
                'dist_to_river','building_density',
                'landuse_type']].describe())

Computing dist_to_building...
✅ dist_to_building done
Computing dist_to_river...
✅ dist_to_river done
Computing building_density...
✅ building_density done
Computing landuse_type...
✅ landuse_type done

✅ ALL OSM FEATURES DONE!
       dist_to_road  dist_to_building  dist_to_river  building_density
count  11225.000000      11225.000000   11225.000000      11225.000000
mean     154.823896       1771.211610     783.918117         50.411403
std      257.486496       1455.232453     820.320163        230.876930
min        0.042840          0.000000       0.367221          0.000000
25%       22.250688        565.050244     231.431493          0.000000
50%       59.510658       1491.693146     513.100210          0.000000
75%      173.854188       2647.629875    1036.344136          2.000000
max     5079.992528      12291.274887    7187.393409       2244.000000


In [31]:
ndvi_path = "../data/raw/hp_ndvi_2023.tif"

with rasterio.open(ndvi_path) as src:
    df_clean['ndvi'] = [val[0] for val in src.sample(coords)]

print("✅ NDVI extracted")
print(df_clean['ndvi'].describe())

✅ NDVI extracted
count    11225.000000
mean         0.410743
std          0.177541
min         -0.143541
25%          0.280950
50%          0.416583
75%          0.540420
max          0.893686
Name: ndvi, dtype: float64


In [33]:
import requests
import time
import numpy as np
import pandas as pd

def get_rainfall_batch(lats, lons, batch_size=20):
    """Fetch rainfall for multiple points using regional average"""
    results = {}
    
    # Round coords to 0.25 degree grid (NASA POWER resolution)
    # This massively reduces unique API calls
    df_coords = pd.DataFrame({'lat': lats, 'lon': lons})
    df_coords['lat_r'] = (df_coords['lat'] * 4).round() / 4
    df_coords['lon_r'] = (df_coords['lon'] * 4).round() / 4
    
    unique_grid = df_coords[['lat_r','lon_r']].drop_duplicates()
    print(f"Reduced to {len(unique_grid)} unique grid points")
    
    for i, (_, row) in enumerate(unique_grid.iterrows()):
        key = (row['lat_r'], row['lon_r'])
        try:
            url = "https://power.larc.nasa.gov/api/temporal/daily/point"
            params = {
                "parameters": "PRECTOTCORR",
                "community": "RE",
                "longitude": row['lon_r'],
                "latitude": row['lat_r'],
                "start": "20230101",
                "end": "20231231",
                "format": "JSON"
            }
            response = requests.get(url, params=params, timeout=30)
            data = response.json()
            rain = pd.Series(
                list(data['properties']['parameter']['PRECTOTCORR'].values())
            )
            results[key] = (
                round(rain.rolling(7).sum().mean(), 3),
                round(rain.rolling(15).sum().mean(), 3)
            )
        except:
            results[key] = (np.nan, np.nan)
        
        if i % 50 == 0:
            print(f"Progress: {i}/{len(unique_grid)}")
        time.sleep(0.3)
    
    return results, df_coords

# Run batch fetch
print("Starting rainfall fetch...")
rainfall_dict, df_coords = get_rainfall_batch(
    df_clean['lat'].values,
    df_clean['lon'].values
)

# Map back using rounded grid coordinates
df_clean['lat_r'] = (df_clean['lat'] * 4).round() / 4
df_clean['lon_r'] = (df_clean['lon'] * 4).round() / 4

df_clean['rainfall_7day'] = df_clean.apply(
    lambda r: rainfall_dict.get(
        (r['lat_r'], r['lon_r']), (np.nan, np.nan)
    )[0], axis=1
)
df_clean['rainfall_15day'] = df_clean.apply(
    lambda r: rainfall_dict.get(
        (r['lat_r'], r['lon_r']), (np.nan, np.nan)
    )[1], axis=1
)

# Clean up temp columns
df_clean.drop(['lat_r', 'lon_r'], axis=1, inplace=True)

print("✅ Rainfall done!")
print(df_clean[['rainfall_7day','rainfall_15day']].describe())

# Save immediately
df_clean.to_csv("../data/processed/features_progress.csv", index=False)
print("✅ Saved!")


Starting rainfall fetch...
Reduced to 74 unique grid points
Progress: 0/74
Progress: 50/74
✅ Rainfall done!
       rainfall_7day  rainfall_15day
count   11225.000000    11225.000000
mean       29.665374       64.872547
std         4.307510        9.428000
min         7.944000       17.331000
25%        27.205000       59.479000
50%        31.966000       69.861000
75%        32.704000       71.527000
max        35.083000       76.743000
✅ Saved!


In [34]:
final_cols = [
    'lat', 'lon', 'District', 'TEHSIL',
    'elevation', 'slope', 'aspect', 'roughness',
    'dist_to_road', 'dist_to_building', 'dist_to_river',
    'building_density', 'landuse_type',
    'ndvi', 'rainfall_7day', 'rainfall_15day',
    'label', 'Exposure'
]

df_final = df_clean[final_cols].copy()
df_final.to_csv("../data/processed/final_dataset.csv", index=False)

print(f"✅ Final dataset saved!")
print(f"Shape: {df_final.shape}")
print(f"\nNull counts:\n{df_final.isnull().sum()}")
print(f"\nPreview:\n{df_final.head()}")

✅ Final dataset saved!
Shape: (11225, 18)

Null counts:
lat                 0
lon                 0
District            0
TEHSIL              0
elevation           0
slope               0
aspect              0
roughness           0
dist_to_road        0
dist_to_building    0
dist_to_river       0
building_density    0
landuse_type        0
ndvi                0
rainfall_7day       0
rainfall_15day      0
label               0
Exposure            0
dtype: int64

Preview:
         lat      lon District        TEHSIL  elevation      slope  \
0  31.808141  76.1385   KANGRA  DERA GOPIPUR        895  12.272894   
1  31.660223  77.3072    KULLU        BANJAR       1424  26.187000   
2  31.459956  77.6727   SHIMLA        RAMPUR       1127  26.800829   
3  31.077288  77.1815   SHIMLA        SHIMLA       1965  37.912567   
4  30.558009  77.2922  SIRMAUR         NAHAN        877  20.887669   

       aspect  roughness  dist_to_road  dist_to_building  dist_to_river  \
0  131.987213       17.0     